# Finding all the cities we need to check

In [ ]:
import requests
import pandas as pd

FILE_PATH = "PeakTempsTop100_old.csv"
df_weather = pd.read_csv(FILE_PATH)
# Get unique cities
cities = df_weather["city"].unique().tolist()

# Replaces "ae", "oe" and "ue" with "ä", "ö", "ü" respectively
def replace_umlauts(text):
    text = text.replace("ae", "ä")
    text = text.replace("oe", "ö")
    text = text.replace("ue", "ü")
    return text

# Replacing 
unique_cities = []
for city in cities:
    if city != "Munich":
        unique_cities.append(replace_umlauts(city))
    else:
        unique_cities.append("München")

# Read the stations CSV and get unique cities
df_stations = pd.read_csv("stations.csv")
station_cities = set(df_stations["city"].unique())
cities_to_check = []

# Finding all the cities that have stations.
# Some of the cities are actually areas of other cities, we ignore those.
for city in unique_cities:
    if city in station_cities:
        cities_to_check.append(city)

cities_to_check
# Save the cities for later use
df = pd.DataFrame(cities_to_check, columns = ["city"])
df.to_csv("top_100_cities.csv", index = False)

#### Finding all the relevant stations per city

In [2]:
# Reading the stations file
df_stations = pd.read_csv("stations.csv")

# Create the dictionary of all stations per city
stations = {
    city: df_stations[df_stations['city'] == city]['id'].tolist()
    for city in cities_to_check
}
stations

{'Berlin': [125,
  126,
  150,
  152,
  154,
  155,
  156,
  160,
  168,
  170,
  171,
  172,
  174,
  176,
  180,
  181,
  182,
  184,
  185,
  189,
  190,
  191,
  192,
  193,
  194,
  195,
  196,
  197,
  198,
  199,
  200,
  201,
  202,
  203,
  204,
  205,
  206,
  207,
  208,
  209,
  210,
  211,
  9424,
  9425,
  9426,
  9427,
  9428,
  9430,
  9431,
  9432,
  9433,
  9434,
  9435,
  9436,
  9474,
  9475,
  9476,
  9477,
  9478,
  9479,
  9480,
  9556,
  10347,
  10355,
  10356,
  10357,
  10358,
  10359,
  144,
  146,
  10379,
  10384,
  121,
  127,
  129,
  130,
  131,
  145,
  151,
  173,
  175,
  183,
  9486,
  10381,
  10382,
  122,
  153,
  161,
  164,
  10377,
  10383,
  112,
  113,
  114,
  116,
  117,
  118,
  119,
  120,
  123,
  128,
  132,
  133,
  134,
  135,
  136,
  139,
  140,
  141,
  142,
  147,
  157,
  158,
  162,
  165,
  166,
  186,
  188,
  10378,
  10380,
  10385,
  10386,
  10387,
  177,
  178,
  179,
  137,
  115,
  124,
  138,
  143,
  148,
  149,
  15

#### Per city, request the PM10 concentrations for every day

In [ ]:
url = "https://luftdaten.umweltbundesamt.de/api/air-data/v4/measures/json"
# Search parameters
# Component 1 = PM10
# Component 9 = PM2.5
params = {
    "date_from": "2016-01-01",
    "date_to": "2025-12-31",
    "time_from": 0,
    "time_to": 23,
    "component": 1,
    "scope" : 1
}

measures_pm10 = {}

# Request all the measurements for each city
for city in stations:
    measures_pm10[city] = []
    for id in stations[city]:
        params["station"] = id
        r = requests.get(url, params=params, timeout=30)
        measures_pm10[city].append(r.json())
measures_pm10

#### Saving PM10 data for the cities

In [78]:
for city_name in measures_pm10.keys():
    values = []
    for result in measures_pm10[city_name]:
        # Only check non-empty values
        if result["data"] != {}:
            # Checking the measurements of each station
            for station in result["data"].keys():
                # Checking all the measurements
                for start_time in result["data"][station]:
                    data_list = result["data"][station][start_time]
                    component_id, scope, value, end_time, index = data_list
                    values.append({
                        "city": city_name,
                        "station": station,
                        "start_time": start_time,
                        "end_time": end_time,
                        "value": value
                    })
    # Save data for each city
    df = pd.DataFrame(values)
    df.to_csv("Measurements/PM10/" + city_name + "PM10.csv", index = False)

#### Per city, request the PM2.5 concentrations for every day

In [ ]:
url = "https://luftdaten.umweltbundesamt.de/api/air-data/v4/measures/json"
# Search parameters
# Component 1 = PM10
# Component 9 = PM2.5
params = {
    "date_from": "2016-01-01",
    "date_to": "2025-12-31",
    "time_from": 0,
    "time_to": 23,
    "component": 9,
    "scope" : 1
}

measures_pm25 = {}

# Request all the measurements for each city
for city in stations:
    measures_pm25[city] = []
    for id in stations[city]:
        params["station"] = id
        r = requests.get(url, params=params, timeout=30)
        measures_pm25[city].append(r.json())
measures_pm25

{'Berlin': [{'request': {'component': '9',
    'scope': '1',
    'station': '169',
    'date_from': '2016-01-01',
    'date_to': '2025-12-31',
    'time_from': '15:00:00',
    'time_to': '23:00:00',
    'recent': False,
    'index': 'id',
    'lang': 'en',
    'datetime_from': '2016-01-01 14:00:00',
    'datetime_to': '2025-12-31 22:00:00'},
   'indices': {'data': {'station id': {'date start': ['component id',
       'scope id',
       'value',
       'date end',
       'index']}}},
   'data': {}}],
 'Hamburg': [{'request': {'component': '9',
    'scope': '1',
    'station': '849',
    'date_from': '2016-01-01',
    'date_to': '2025-12-31',
    'time_from': '15:00:00',
    'time_to': '23:00:00',
    'recent': False,
    'index': 'id',
    'lang': 'en',
    'datetime_from': '2016-01-01 14:00:00',
    'datetime_to': '2025-12-31 22:00:00'},
   'indices': {'data': {'station id': {'date start': ['component id',
       'scope id',
       'value',
       'date end',
       'index']}}},
   'da

#### Saving PM2.5 data for the cities

In [80]:
for city_name in measures_pm25.keys():
    values = []
    for result in measures_pm25[city_name]:
        # Only check non-empty values
        if result["data"] != {}:
            # Checking the measurements of each station
            for station in result["data"].keys():
                # Checking all the measurements
                for start_time in result["data"][station]:
                    data_list = result["data"][station][start_time]
                    component_id, scope, value, end_time, index = data_list
                    values.append({
                        "city": city_name,
                        "station": station,
                        "start_time": start_time,
                        "end_time": end_time,
                        "value": value
                    })
    # Save data for each city
    df = pd.DataFrame(values)
    df.to_csv("Measurements/PM25/" + city_name + "PM25.csv", index = False)